In [ ]:
# ============================================================
# Cell 1: Imports, Configuration & Constants
# ============================================================
import os
import time
import shutil
from datetime import datetime
from urllib.parse import urlparse

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# Optional: webdriver-manager for automatic ChromeDriver resolution
try:
    from webdriver_manager.chrome import ChromeDriverManager
    HAS_WDM = True
except Exception:
    HAS_WDM = False

# ---------- Report Configuration ----------
# Each entry defines a Looker report URL and its desired filename prefix
REPORTS_CONFIG = [
    {
        "url": "https://console.vap.expedia.com/analytics-console-user-interface/reports/personal_reports/new_look_excel_en",
        "prefix": "New Look Excel data_EN"
    },
    {
        "url": "https://console.vap.expedia.com/analytics-console-user-interface/reports/personal_reports/delayed_closure_short_chats_agent_disconnect",
        "prefix": "Delayed Closure, Short Chats, Agent Disconnect_"
    },
    {
        "url": "https://console.vap.expedia.com/analytics-console-user-interface/reports/personal_reports/survey_dump_with_feedback_ir_agentexp",
        "prefix": "Survey Dump with Feedback, IR, agentExp"
    },
    {
        "url": "https://console.vap.expedia.com/analytics-console-user-interface/reports/personal_reports/t3_cnx",
        "prefix": "T3 CNX"
    },
    # {
    #     "url": "https://console.vap.expedia.com/analytics-console-user-interface/reports/personal_reports/report_url_5",
    #     "prefix": "Report_Name_5"
    # }
]

# ---------- Path Configuration ----------
PROFILE_DIR  = r"C:/temp/new_chrome_profile"
CHROMEDRIVER = r"C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/CAPTURE/chromedriver-win64/chromedriver.exe"
DOWNLOAD_DIR = r"C:/temp/expedia_downloads"

# ---------- Timing Constants ----------
ACTION_DELAY       = 1     # Short pause between UI actions (seconds)
PAGE_LOAD_TIMEOUT  = 180   # Max wait for Okta login and initial page load
WAIT_AFTER_OPEN    = 180   # Hard-wait for Looker data to fully render (seconds)
TICK_SECONDS       = 60    # Download monitor polling interval
MAX_WAIT_MINUTES   = 30    # Maximum download wait time (minutes)

# ---------- Other Constants ----------
PERIOD   = datetime.now().strftime("%Y-%m")           # Current year-month used in filenames
EXT_DONE = (".csv", ".zip", ".xlsx")                  # File extensions that indicate a completed download

# SVG path data prefix used to locate the gear/settings icon in Looker UI
GEAR_PATH_PREFIX = (
    "M19.43 12.98c.04-.32.07-.64.07-.98 0-.34-.03-.66-.07-.98l2.11-1.65c.19-.15"
    ".24-.42.12-.64l-2-3.46a.5.5 0 00-.61-.22l-2.49 1c-.52-.4-1.08-.73-1.69-.98l-."
    "38-2.65A.488.488 0 0014 2h-4c-.25 0-.46.18-.49.42l-.38 2.65c-.61.25-1.17.59-1."
    "69.98l-2.49-1a.566.566 0 00-.18-.03c-.17 0-.34.09-.43.25l-2 3.46c-.13.22-.07."
    "49.12.64l2.11 1.65c-.04.32-.07.65-.07.98 0 .33.03.66.07.98l-2.11 1.65c-.19.15-."
    "24.42-.12.64l2 3.46a.5.5 0 00.61.22l2.49-1c.52.4 1.08.73 1.69.98l.38 2.65c.03."
    "24.24.42.49.42h4c.25 0 .46-.18.49-.42l.38-2.65c.61-.25 1.17-.59 1.69-.98l2.49 1c."
    "06.02.12.03.18.03.17 0 .34-.09.43-.25l2-3.46c.12-.22.07-.49-.12-.64l-2.11-1.65zm-"
    "1.98-1.71c.04.31.05.52.05.73 0 .21-.02.43-.05.73l-.14 1.13.89.7 1.08.84-.7 1.21-"
    "1.27-.51-1.04-.42-.9.68c-.43.32-.84.56-1.25.73l-1.06.43-.16 1.13-.2 1.35h-1.4l-."
    "19-1.35-.16-1.13-1.06-.43c-.43-.18-.83-.41-1.23-.71l-.91-.7-1.06.43-1.27.51-.7-"
    "1.21 1.08-.84.89-.7-.14-1.13c-.03-.31-.05-.54-.05-.74s.02-.43.05-.73l.14-1.13-."
    "89-.7-1.08-.84.7-1.21 1.27.51 1.04.42.9-.68c.43-.32.84-.56 1.25-.73l1.06-.43.16-"
    "1.13.2-1.35h1.39l.19 1.35.16 1.13 1.06.43c.43.18.83.41 1.23.71l.91.7 1.06-.43 1.27-"
    ".51.7 1.21-1.07.85-.89.7.14 1.13zM12 8c-2.21 0-4 1.79-4 4s1.79 4 4 4 4-1.79 4-4-1."
    "79-4-4-4zm0 6c-1.1 0-2-.9-2-2s.9-2 2-2 2 .9 2 2-.9 2-2 2z"
)

def pause():
    """Short pause between UI actions to allow the page to respond."""
    time.sleep(ACTION_DELAY)

In [ ]:
# ============================================================
# Cell 2: WebDriver Setup & Shadow DOM / iframe Utilities
# ============================================================

def build_driver():
    """Initialize and return a configured Chrome WebDriver instance."""
    # Ensure the download directory exists with an absolute path (Windows-safe)
    download_path = os.path.abspath(DOWNLOAD_DIR)
    os.makedirs(download_path, exist_ok=True)

    o = Options()
    o.add_argument(f"--user-data-dir={PROFILE_DIR}")
    o.add_argument("--profile-directory=Default")
    o.add_argument("--start-maximized")
    o.add_argument("--disable-session-crashed-bubble")  # Suppress crash restore prompts
    o.add_argument("--hide-crash-restore-bubble")       # Hide 'restore tabs' banner
    o.add_argument("--disable-restore-session-state")   # Do not restore previous session tabs

    # Required for Looker embedded content to load correctly
    o.add_argument("--disable-web-security")
    o.add_argument("--allow-running-insecure-content")

    o.add_experimental_option(
        "prefs",
        {
            "download.default_directory":                              download_path,
            "download.prompt_for_download":                           False,
            "download.directory_upgrade":                             True,
            "profile.default_content_settings.popups":                0,
            "profile.default_content_setting_values.automatic_downloads": 1,
            "profile.cookie_controls_mode":                           0,      # CRITICAL: allows third-party Looker cookies
            "safebrowsing.enabled":                                   False,
            "safebrowsing.disable_download_protection":               True,
        },
    )

    # Resolve ChromeDriver using local path, webdriver-manager, or Selenium Manager (fallback)
    if os.path.exists(CHROMEDRIVER):
        print("Using local ChromeDriver path.")
        driver = webdriver.Chrome(service=Service(CHROMEDRIVER), options=o)
    elif HAS_WDM:
        print("Using WebDriverManager to resolve ChromeDriver.")
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=o)
    else:
        print("Using Selenium Manager (built-in).")
        driver = webdriver.Chrome(options=o)

    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    driver.set_script_timeout(180)

    # Enable CDP-level download behavior (best-effort; may fail on some versions)
    try:
        driver.execute_cdp_cmd(
            "Page.setDownloadBehavior",
            {"behavior": "allow", "downloadPath": download_path},
        )
    except Exception:
        pass

    return driver


# JavaScript that queries an element across Shadow DOM roots recursively
_DEEP_QUERY_JS = r"""
const sel = arguments[0];
function deepQuery(root) {
  const el = root.querySelector(sel);
  if (el) return el;
  const all = root.querySelectorAll('*');
  for (const n of all) if (n.shadowRoot) {
    const f = deepQuery(n.shadowRoot);
    if (f) return f;
  }
  return null;
}
return deepQuery(document);
"""


def _for_all_contexts(driver, fn):
    """Run fn() in the current frame context; recurse into child iframes."""
    if fn():
        return True
    for fr in driver.find_elements(By.TAG_NAME, "iframe"):
        try:
            driver.switch_to.frame(fr)
            if _for_all_contexts(driver, fn):
                return True
            driver.switch_to.parent_frame()
        except Exception:
            try:
                driver.switch_to.parent_frame()
            except Exception:
                pass
    return False


def deep_try(driver, fn, timeout=20, poll=0.3):
    """Poll fn() across all iframes and shadow roots until it returns True or times out."""
    end = time.time() + timeout
    while time.time() < end:
        driver.switch_to.default_content()
        if _for_all_contexts(driver, fn):
            return True
        time.sleep(poll)
        driver.switch_to.default_content()
    return False


def deep_get(driver, css, timeout=20):
    """Find and return an element by CSS selector, traversing Shadow DOM and iframes."""
    holder = {"el": None}

    def fn():
        el = driver.execute_script(_DEEP_QUERY_JS, css)
        if el:
            holder["el"] = el
            return True
        return False

    if not deep_try(driver, fn, timeout):
        raise TimeoutException(f"Deep element not found: {css}")
    return holder["el"]


def deep_click_text(driver, text, timeout=15):
    """Find and click the first element whose text matches exactly, traversing Shadow DOM and iframes."""
    js = r"""
const target = arguments[0];
function deepFindByText(root) {
  const all = root.querySelectorAll('*');
  for (const el of all) {
    if (el.shadowRoot) {
      const f = deepFindByText(el.shadowRoot);
      if (f) return f;
    }
    const t = (el.innerText || el.textContent || '').trim();
    if (!t) continue;
    if (t === target) {
      return el.closest('[role="option"],li,button,[role="button"],a,div,span') || el;
    }
  }
  return null;
}
return deepFindByText(document);
"""
    holder = {"el": None}

    def fn():
        el = driver.execute_script(js, text)
        if el:
            holder["el"] = el
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
            try:
                el.click()
            except Exception:
                driver.execute_script("arguments[0].click();", el)
            return True
        return False

    if not deep_try(driver, fn, timeout):
        raise TimeoutException(f"Deep text not found: {text}")
    pause()
    return holder["el"]


def deep_find_and_switch(driver, css, timeout=20):
    """Find an element across iframes/Shadow DOM and leave the driver switched into that frame context."""
    end = time.time() + timeout

    def search_here():
        el = driver.execute_script(_DEEP_QUERY_JS, css)
        if el:
            return el
        for fr in driver.find_elements(By.TAG_NAME, "iframe"):
            try:
                driver.switch_to.frame(fr)
                found = search_here()
                if found:
                    return found
                driver.switch_to.parent_frame()
            except Exception:
                try:
                    driver.switch_to.parent_frame()
                except Exception:
                    pass
        return None

    while time.time() < end:
        driver.switch_to.default_content()
        el = search_here()
        if el:
            return el
        time.sleep(0.3)
        driver.switch_to.default_content()

    raise TimeoutException(f"Deep element not found: {css}")

In [ ]:
# ============================================================
# Cell 3: Login & Navigation
# ============================================================

def check_and_login(driver, url, path_fragment, timeout=PAGE_LOAD_TIMEOUT):
    """Navigate to the report URL, handle Okta SSO login if required, then wait for the page to fully load."""
    print(f"Navigating to: {url}")
    driver.get(url)
    WebDriverWait(driver, timeout).until(lambda d: d.execute_script("return document.readyState") == "complete")
    print("Base page loaded.")
    pause()

    # Attempt Okta login; if already authenticated, the sign-in button won't appear
    try:
        btn = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[data-testid="console-okta-sign-in"]'))
        )
        print("Sign-in required detected. Clicking sign-in...")
        btn.click()
        pause()

        # Attempt to check 'Remember Me' checkbox
        try:
            lab = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, 'label[for="input36"][data-se-for-name="rememberMe"]'))
            )
            lab.click()
            pause()
        except TimeoutException:
            pass

        # Click 'Next' to proceed through Okta MFA flow
        try:
            nxt = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, 'input.button.button-primary[type="submit"][value="Next"]')
                )
            )
            nxt.click()
            print("Clicked 'Next'. Waiting for Okta to process token...")
            time.sleep(5)
            pause()
        except TimeoutException:
            pass

        # Dismiss any browser alert dialogs that may appear during SSO
        try:
            WebDriverWait(driver, 5).until(EC.alert_is_present())
            driver.switch_to.alert.accept()
            print("Accepted alert.")
            pause()
        except Exception:
            pass

        print("Login flow processed. Waiting for target dashboard...")
        pause()

    except TimeoutException:
        print("No Okta sign-in required. Session is already active.")

    # Confirm the URL matches the expected report path
    try:
        WebDriverWait(driver, PAGE_LOAD_TIMEOUT).until(EC.url_contains(path_fragment))
    except TimeoutException:
        print(f"Warning: URL did not contain '{path_fragment}' within {PAGE_LOAD_TIMEOUT}s. Continuing anyway...")

    WebDriverWait(driver, PAGE_LOAD_TIMEOUT).until(lambda d: d.execute_script("return document.readyState") == "complete")
    print("Page ready.")

    # Hard-wait to ensure all Looker data is rendered before interacting
    print(f"Hard-waiting {WAIT_AFTER_OPEN}s to ensure Looker data is fully rendered...")
    time.sleep(WAIT_AFTER_OPEN)
    print("Hard-wait finished. Proceeding with UI interactions.")
    pause()

In [ ]:
# ============================================================
# Cell 4: Download Modal Interaction & File Monitoring
# ============================================================

def click_gear_by_path(driver, d_prefix=GEAR_PATH_PREFIX, timeout=25):
    """Locate and click the gear icon in Looker by matching its SVG path data."""
    js = r"""
const dPrefix = arguments[0];
function deepWalk(root) {
  const paths = root.querySelectorAll('path[d]');
  for (const p of paths) {
    const d = p.getAttribute('d') || '';
    if (d && (d === dPrefix || d.startsWith(dPrefix))) {
      return p.closest('button,[role="button"],a,div,span') || p;
    }
  }
  const all = root.querySelectorAll('*');
  for (const el of all) {
    if (el.shadowRoot) {
      const f = deepWalk(el.shadowRoot);
      if (f) return f;
    }
  }
  return null;
}
return deepWalk(document);
"""
    holder = {"el": None}

    def fn():
        el = driver.execute_script(js, d_prefix)
        if el:
            holder["el"] = el
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
            try:
                el.click()
            except Exception:
                driver.execute_script("arguments[0].click();", el)
            return True
        return False

    if not deep_try(driver, fn, timeout):
        raise TimeoutException("Gear path not found across frames/shadow DOM.")
    print("Clicked gear icon.")
    pause()


def open_download_modal(driver):
    """Open the Looker download modal — first try direct text click, then via gear menu."""
    try:
        deep_click_text(driver, "Download", timeout=6)
        print("Opened Download modal directly.")
        return
    except TimeoutException:
        pass
    click_gear_by_path(driver)
    deep_click_text(driver, "Download", timeout=10)
    print("Opened Download modal via gear.")


def set_download_format_to_csv(driver):
    """Set the export format dropdown in the Download modal to CSV."""
    btn = deep_get(driver, "#listbox-input-qr-export-modal-format", timeout=20)
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
    try:
        btn.click()
    except Exception:
        driver.execute_script("arguments[0].click();", btn)
    pause()

    # Select CSV from the dropdown options
    try:
        deep_click_text(driver, "CSV", timeout=8)
        print("Selected CSV format.")
    except TimeoutException:
        # Fallback: type and confirm via keyboard
        btn.send_keys("CSV")
        btn.send_keys(Keys.ENTER)
        pause()
        print("Entered CSV format via keyboard.")

    ActionChains(driver).send_keys(Keys.ESCAPE).perform()
    time.sleep(1)

    # Validate that the format was correctly set
    btn2 = deep_get(driver, "#listbox-input-qr-export-modal-format", timeout=10)
    if (btn2.get_attribute("value") or "").strip().upper() != "CSV":
        raise TimeoutException("Format not set to CSV.")
    print("Format is successfully set to CSV.")
    pause()


def set_export_filename(driver, name_prefix, period):
    """Type the desired filename into the export filename field in the Download modal."""
    box = deep_get(driver, "#qr-export-modal-custom-filename", timeout=20)
    filename = f"{name_prefix} {period}"

    try:
        # Clear existing content and type new filename
        box.click(); pause()
        box.send_keys(Keys.CONTROL, "a"); pause()
        box.send_keys(Keys.BACK_SPACE); pause()
        box.send_keys(filename); pause()
    except Exception:
        # Fallback: use JavaScript to set the value and trigger React's change events
        driver.execute_script(
            """
const el = arguments[0], val = arguments[1];
const setter = Object.getOwnPropertyDescriptor(HTMLInputElement.prototype, 'value').set;
setter.call(el, val);
el.dispatchEvent(new Event('input', {bubbles:true}));
el.dispatchEvent(new Event('change', {bubbles:true}));
""", box, filename)
        pause()

    if (box.get_attribute("value") or "").strip() != filename:
        raise TimeoutException("Filename not applied properly.")
    print(f"Set filename to: {filename}")
    pause()


def select_unformatted_and_all_results(driver):
    """Select 'Unformatted' data option and 'All results' in the Download modal."""
    time.sleep(1)
    deep_click_text(driver, "Unformatted (no rounding, special characters, etc.)", timeout=15)
    pause()
    deep_click_text(driver, "All results", timeout=10)
    pause()
    _deep_find_enabled_download(driver, timeout=20)  # Wait until Download button becomes active
    print("Selected 'Unformatted' and 'All results' options.")
    pause()


def _absolute_center_in_page(driver, el):
    """Calculate the absolute (x, y) screen coordinates of an element's center, even inside nested iframes."""
    return driver.execute_script(
        """
const el = arguments[0];
const r = el.getBoundingClientRect();
let x = (r.left + r.right) / 2;
let y = (r.top + r.bottom) / 2;
let w = window;
for (let i = 0; i < 15; i++) {
  if (w === w.parent) break;
  try {
    const fr = w.frameElement;
    if (!fr) break;
    const rf = fr.getBoundingClientRect();
    x += rf.left;
    y += rf.top;
    w = w.parent;
  } catch (e) { break; }
}
return {x, y};
""", el)


def _cdp_click_at(driver, x, y):
    """Dispatch a CDP mouse click at the given absolute (x, y) page coordinates."""
    driver.execute_cdp_cmd("Input.dispatchMouseEvent", {"type": "mouseMoved",   "x": x, "y": y, "buttons": 1})
    driver.execute_cdp_cmd("Input.dispatchMouseEvent", {"type": "mousePressed", "x": x, "y": y, "button": "left", "buttons": 1, "clickCount": 1})
    driver.execute_cdp_cmd("Input.dispatchMouseEvent", {"type": "mouseReleased","x": x, "y": y, "button": "left", "buttons": 1, "clickCount": 1})


def _deep_find_enabled_download(driver, timeout=30, poll=0.2):
    """Poll until the Download button is visible and not disabled; raise on timeout."""
    end = time.time() + timeout
    last_err = None
    while time.time() < end:
        try:
            driver.switch_to.default_content()
            btn = deep_find_and_switch(driver, "button#qr-export-modal-download, #qr-export-modal-download", timeout=5)
            dis  = (btn.get_attribute("disabled")      or "").strip().lower()
            aria = (btn.get_attribute("aria-disabled") or "").strip().lower()
            if btn.is_displayed() and dis in ("", "false", None) and aria in ("", "false", None):
                return btn, True
        except Exception as e:
            last_err = e
        time.sleep(poll)
    raise TimeoutException(f"Download button not enabled in time: {last_err}")


def click_modal_download(driver, expected_filename):
    """
    Attempt to click the Download button using multiple strategies:
    ActionChains -> JS click -> CDP mouse events.
    Returns the timestamp when the download was detected.
    """
    from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException, JavascriptException

    def _find_btn(timeout=8):
        driver.switch_to.default_content()
        return deep_find_and_switch(driver, "button#qr-export-modal-download, #qr-export-modal-download", timeout=timeout)

    def _has_started_download():
        """Check for a .crdownload file matching the expected filename prefix."""
        try:
            pref = expected_filename.lower()
            for n in os.listdir(DOWNLOAD_DIR):
                if n.lower().startswith(pref) and n.endswith(".crdownload"):
                    return True
        except Exception:
            pass
        return False

    def _prep(btn_el):
        """Force-enable the button and scroll it into view via JavaScript."""
        try:
            driver.execute_script("""
                const b = arguments[0];
                b.removeAttribute('disabled');
                b.setAttribute('aria-disabled','false');
                b.scrollIntoView({block:'center'});
                try { b.focus({preventScroll:true}); } catch(e) {}
            """, btn_el)
        except Exception:
            pass

    def _abs_center_retry(btn_el, attempts=3):
        """Get absolute center coordinates with retry on stale element errors."""
        last_err = None
        for _ in range(attempts):
            try:
                return _absolute_center_in_page(driver, btn_el)
            except (StaleElementReferenceException, NoSuchElementException, JavascriptException) as e:
                last_err = e
                try:
                    btn_el = _find_btn(timeout=5)
                except Exception:
                    time.sleep(0.2)
        if last_err:
            raise last_err
        return {"x": 0, "y": 0}

    def _hover_abs(absx, absy):
        """Move the CDP mouse cursor to the given absolute coordinates."""
        try:
            driver.execute_cdp_cmd("Input.dispatchMouseEvent", {"type": "mouseMoved", "x": float(absx), "y": float(absy), "buttons": 1})
        except Exception:
            pass

    def _clear_overlay_at(absx, absy, restore_ms=800):
        """Temporarily disable pointer-events on any overlay covering the button, then restore."""
        try:
            driver.switch_to.default_content()
            driver.execute_script("""
                (function(x,y,ms){
                  const el = document.elementFromPoint(x,y);
                  if(!el) return;
                  const old = el.style.pointerEvents;
                  el.style.pointerEvents = 'none';
                  setTimeout(()=>{ el.style.pointerEvents = old; }, ms);
                })(arguments[0], arguments[1], arguments[2]);
            """, float(absx), float(absy), int(restore_ms))
        except Exception:
            pass
        try:
            deep_find_and_switch(driver, "button#qr-export-modal-download, #qr-export-modal-download", timeout=2)
        except Exception:
            pass

    # Try up to 5 rounds of click strategies
    rounds = 5
    for r in range(rounds):
        try:
            btn = _find_btn(timeout=10)
        except Exception:
            time.sleep(0.2)
            continue

        _prep(btn)
        coords = _abs_center_retry(btn, attempts=3)
        absx, absy = float(coords["x"]), float(coords["y"])
        _clear_overlay_at(absx, absy)
        _hover_abs(absx, absy)
        time.sleep(0.08)

        # Strategy 1: ActionChains click
        try:
            btn = _find_btn(timeout=4)
            ActionChains(driver).move_to_element(btn).pause(0.05).click().perform()
        except Exception:
            pass
        time.sleep(0.12)
        if _has_started_download():
            print("Clicked Download -> detected .crdownload via ActionChains.")
            return time.time()

        # Strategy 2: JavaScript click
        try:
            btn = _find_btn(timeout=4)
            driver.execute_script("arguments[0].click();", btn)
        except Exception:
            pass
        time.sleep(0.15)
        if _has_started_download():
            print("Clicked Download -> detected .crdownload via JavaScript.")
            return time.time()

        # Strategy 3: CDP mouse event with slight coordinate jitter
        try:
            driver.switch_to.default_content()
            for dx, dy in [(0, 0), (3, 2), (-3, -2), (2, -3), (-2, 3)]:
                _cdp_click_at(driver, absx + dx, absy + dy)
                time.sleep(0.08)
        except Exception:
            pass

        try:
            deep_find_and_switch(driver, "button#qr-export-modal-download, #qr-export-modal-download", timeout=2)
        except Exception:
            pass

        if _has_started_download():
            print("Clicked Download -> detected .crdownload via CDP trigger.")
            return time.time()
        time.sleep(0.6)

    print("Clicked Download (best-effort completed).")
    return time.time()


def _list_matching_files(dirpath, base_prefix, since_ts=None):
    """List files in dirpath whose names start with base_prefix, optionally filtered by modification time."""
    base = (base_prefix or "").lower()
    try:
        names = os.listdir(dirpath)
    except FileNotFoundError:
        return []
    paths = []
    for n in names:
        if not n.lower().startswith(base):
            continue
        p = os.path.join(dirpath, n)
        try:
            if since_ts is None or os.path.getmtime(p) >= since_ts - 2:
                paths.append(p)
        except FileNotFoundError:
            pass
    return paths


def wait_download_minute_ticks(dirpath, expected_prefix, since_ts, tick_seconds=60, max_minutes=30):
    """Poll the download directory at regular intervals and return the completed file path, or None on timeout."""
    total_ticks = max(1, int(max_minutes * 60 / tick_seconds))
    last_size = None

    def _human_size(n):
        """Convert a byte count to a human-readable string (e.g., '1.2MB')."""
        try:
            n = float(n)
        except Exception:
            return "?"
        for unit in ("B", "KB", "MB", "GB", "TB"):
            if n < 1024.0:
                return f"{n:.1f}{unit}"
            n /= 1024.0
        return f"{n:.1f}PB"

    print(f"[MONITOR] Waiting up to {max_minutes} minutes, checking every {tick_seconds}s...")

    # Check if a completed file already exists before starting the poll loop
    any_paths = _list_matching_files(dirpath, expected_prefix, since_ts=None)
    any_done  = [p for p in any_paths if not p.endswith(".crdownload") and os.path.splitext(p)[1].lower() in EXT_DONE]
    if any_done:
        target = max(any_done, key=lambda p: os.path.getmtime(p))
        print(f"Found existing file: {os.path.basename(target)} ({_human_size(os.path.getsize(target))})")
        return target

    for i in range(1, total_ticks + 1):
        recent    = _list_matching_files(dirpath, expected_prefix, since_ts=since_ts)
        any_paths = recent or _list_matching_files(dirpath, expected_prefix, since_ts=None)

        inprog = [p for p in any_paths if p.endswith(".crdownload")]
        done   = [p for p in any_paths if not p.endswith(".crdownload") and os.path.splitext(p)[1].lower() in EXT_DONE]

        # File finished downloading (no remaining .crdownload)
        if done and not inprog:
            target = max(done, key=lambda p: os.path.getmtime(p))
            s1 = os.path.getsize(target)
            time.sleep(2)
            s2 = os.path.getsize(target)
            if s1 == s2:  # Confirm file size is stable (write complete)
                print(f"Download finished: {os.path.basename(target)} ({_human_size(s2)})")
                return target

        # File is still downloading
        if inprog:
            cur = max(inprog, key=lambda p: os.path.getmtime(p))
            try:
                sz = os.path.getsize(cur)
            except FileNotFoundError:
                sz = None
            if sz is not None:
                if last_size is not None and sz > last_size:
                    delta = sz - last_size
                    print(f"[{i:02d}/{total_ticks}] {_human_size(sz)} downloaded (+{_human_size(delta)} since last check)")
                else:
                    print(f"[{i:02d}/{total_ticks}] {_human_size(sz)} downloaded (no change)")
                last_size = sz
        else:
            if not any_paths:
                print(f"[{i:02d}/{total_ticks}] No matching files yet...")
            else:
                names = ", ".join(os.path.basename(p) for p in any_paths)
                print(f"[{i:02d}/{total_ticks}] Files present: {names} (waiting for completion)")

        if i < total_ticks:
            time.sleep(tick_seconds)

    return None

In [ ]:
# ============================================================
# Cell 5: Main Execution — Iterate Reports & Download
# ============================================================

# 1. Initialize the Chrome WebDriver
driver = build_driver()

try:
    # Iterate over each report defined in REPORTS_CONFIG
    for i, report in enumerate(REPORTS_CONFIG, start=1):
        target_url  = report["url"]
        name_prefix = report["prefix"]
        path_fragment = urlparse(target_url).path.split("/")[-1]

        print(f"\n{'='*50}")
        print(f"PROCESSING REPORT {i}/{len(REPORTS_CONFIG)}: {name_prefix}")
        print(f"{'='*50}")

        try:
            # 2. Navigate to the report page and handle authentication if needed
            check_and_login(driver, target_url, path_fragment, PAGE_LOAD_TIMEOUT)

            # 3. Interact with the Looker UI to configure and trigger the export
            open_download_modal(driver)
            set_download_format_to_csv(driver)
            set_export_filename(driver, name_prefix, PERIOD)
            select_unformatted_and_all_results(driver)

            # 4. Click the Download button and record the trigger timestamp
            expected_filename = f"{name_prefix} {PERIOD}"
            click_ts = click_modal_download(driver, expected_filename)

            # 5. Monitor the download directory until the file is complete or timeout
            path = wait_download_minute_ticks(
                DOWNLOAD_DIR,
                expected_filename,
                since_ts=click_ts,
                tick_seconds=TICK_SECONDS,
                max_minutes=MAX_WAIT_MINUTES,
            )

            if path:
                print(f"Successfully downloaded: {path}")
            else:
                print(f"Timeout occurred while downloading '{name_prefix}'.")

            # Brief pause before processing the next report
            time.sleep(5)

        except Exception as inner_e:
            print(f"Failed to process report '{name_prefix}'. Error: {inner_e}")
            print("Moving to the next report in queue...")
            continue  # Skip to next report even if current one fails

except Exception as e:
    print(f"A critical error occurred in the main loop: {e}")

finally:
    # Always close the browser after all reports are processed
    print("\nAll reports processed. Closing Chrome.")
    driver.quit()

In [ ]:
# ============================================================
# Cell 6: Move Downloaded Files to Target OneDrive Folders
# ============================================================

# Resolve the user's home directory (cross-platform)
first_glob   = os.path.expanduser("~").replace("\\", "/")
DOWNLOAD_DIR = r"C:/temp/expedia_downloads"

# Base paths for each report category in OneDrive/SharePoint
_BASE = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW"

folder_paths = {
    "input_performance":     f"{_BASE}/NEW_LOOK_EXCEL_EN/new_look_excel_data",
    "input_survey":          f"{_BASE}/SURVEY_EN",
    "input_t3":              f"{_BASE}/T3_EN",
    "input_delayed_closure": f"{_BASE}/DELAYED_CLOSURE",
}

# Map each filename prefix to its destination folder
PREFIX_MAPPING = {
    "New Look Excel data_EN":                            folder_paths["input_performance"],
    "Survey Dump with Feedback, IR, agentExp":           folder_paths["input_survey"],
    "T3 CNX":                                            folder_paths["input_t3"],
    "Delayed Closure, Short Chats, Agent Disconnect":    folder_paths["input_delayed_closure"],
}


def move_downloaded_files():
    """Move completed downloads from DOWNLOAD_DIR to their mapped OneDrive destination folders."""
    print(f"\n{'='*50}")
    print("STARTING TO MOVE DOWNLOADED FILES TO TARGET FOLDERS")
    print(f"{'='*50}")

    if not os.path.exists(DOWNLOAD_DIR):
        print(f"Download directory not found: {DOWNLOAD_DIR}")
        return

    files = os.listdir(DOWNLOAD_DIR)
    if not files:
        print("Download directory is currently empty.")
        return

    success_count = 0
    for file in files:
        # Skip incomplete or temporary files
        if file.endswith(".crdownload") or file.endswith(".tmp"):
            continue

        src_path      = os.path.join(DOWNLOAD_DIR, file)
        target_folder = None

        # Match file to its destination by prefix
        for prefix, folder in PREFIX_MAPPING.items():
            if file.startswith(prefix):
                target_folder = folder
                break

        if target_folder:
            os.makedirs(target_folder, exist_ok=True)
            dst_path = os.path.join(target_folder, file)

            try:
                if os.path.exists(dst_path):  # Overwrite if file already exists at destination
                    os.remove(dst_path)

                shutil.move(src_path, dst_path)
                print(f"Moved: {file}\n   -> ...{target_folder[-40:]}")
                success_count += 1
            except Exception as e:
                print(f"Failed to move file '{file}': {e}")
        else:
            print(f"Skipped (no matching prefix): {file}")

    print(f"{'='*50}")
    print(f"Successfully processed {success_count} file(s).")


move_downloaded_files()